# nb05 — One-Stage Anchor-Based: RetinaNet Baseline (R0)

**Goal.** Train `retinanet_resnet50_fpn_v2` (COCO-pretrained backbone, 9-class head) under the **locked protocol** established in nb02–nb04:

- Preprocessing: `letterbox-1280` (loaded from `chosen_preprocessing.json`)
- Optimizer: SGD, `lr=5e-3`, `momentum=0.9`, `weight_decay=1e-4`
- Scheduler: StepLR (step=3, gamma=0.1)
- Batch size: 2, **1 epoch** (T4 budget)
- Clip-disjoint train/val split
- Full-val COCO evaluation with `iscrowd=1` for ignore regions

**This notebook produces:**
- `retinanet_R0.pth` — baseline checkpoint (upload to a Kaggle dataset)
- `val_preds_retinanet_R0.json` — full-val COCO predictions
- W&B run `retinanet_R0` with training curves, config, and validation predictions

**R1 (data-grounded anchors) lives in nb06.** Do not run it here.
**R2 (focal γ) is dropped** per the compute-quota decision documented in the report.

**Session plan.** This notebook trains **only R0**. After it finishes:
1. Upload `retinanet_R0.pth` and `val_preds_retinanet_R0.json` as a Kaggle dataset.
2. Proceed to nb06 (R1), which mounts that dataset and warm-starts from R0.

In [1]:
import sys, json, time, shutil, os
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision.ops import box_iou
from torchvision.models.detection import retinanet_resnet50_fpn_v2
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.anchor_utils import AnchorGenerator

# ---- Paths -----------------------------------------------------------------
COMP_DIR     = Path("/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026")
DATASET_DIR  = Path("/kaggle/input/datasets/suchok/my-dataset")
OUTPUTS_DIR  = Path("/kaggle/input/datasets/suchok/outputs")            # nb02 artifacts
WEIGHTS_DIR  = Path("/kaggle/input/datasets/suchok/baseline-weights")   # nb03 baseline ckpt
WORK_DIR     = Path("/kaggle/working")

assert COMP_DIR.exists(),    f"missing competition: {COMP_DIR}"
assert DATASET_DIR.exists(), f"missing my-dataset: {DATASET_DIR}"
assert OUTPUTS_DIR.exists(), f"missing outputs: {OUTPUTS_DIR}"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ---- common.py --------------------------------------------------------------
sys.path.insert(0, str(DATASET_DIR))
import common, importlib
importlib.reload(common)
from common import (
    set_seeds, SEED, CLASS_NAMES, NUM_CLASSES,
    load_splits, index_split, filter_by_clips,
    build_fullval_gt,
    evaluate_full_val, save_metrics, save_checkpoint,
    detection_collate, VisDroneDetectionDataset,
    predict_with_strategy, predict_test_and_write_submission,
)

set_seeds(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Speed flags — safe, do not change numerics materially
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False  # already non-deterministic
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass  # older torch, or no CUDA — silently ignore
print("DEVICE:", DEVICE, "| torch:", torch.__version__)

# ---- Locked protocol --------------------------------------------------------
chosen = json.load(open(OUTPUTS_DIR / "chosen_preprocessing.json"))
STRAT  = chosen["strategy"]
SIZE   = int(chosen["size"])
print("Locked preprocessing:", STRAT, SIZE)

train_clips, val_clips = load_splits(OUTPUTS_DIR / "splits")
train_index = filter_by_clips(index_split(COMP_DIR / "train"), train_clips)
val_index   = filter_by_clips(index_split(COMP_DIR / "val"),   val_clips)
print(f"train frames: {len(train_index):,} | val frames: {len(val_index):,}")

TEST_CSV = COMP_DIR / "test_images.csv"

# ---- Full-val GT (cached) ---------------------------------------------------
gt_fullval_ignore = WORK_DIR / "gt_fullval_ignore.json"
for src in (WEIGHTS_DIR, OUTPUTS_DIR):
    if (src / "gt_fullval_ignore.json").exists() and not gt_fullval_ignore.exists():
        shutil.copy(src / "gt_fullval_ignore.json", gt_fullval_ignore)
        print(f"copied gt_fullval_ignore.json from {src}")
if not gt_fullval_ignore.exists():
    build_fullval_gt(val_index, str(gt_fullval_ignore), include_ignore=True)
    print("built gt_fullval_ignore.json")

DEVICE: cuda | torch: 2.11.0+cu130
Locked preprocessing: letterbox 1280
train frames: 24,201 | val frames: 2,846
copied gt_fullval_ignore.json from /kaggle/input/datasets/suchok/baseline-weights


## 1. Bootstrap

Load `common.py`, recover the locked preprocessing decision and clip split, build the full-val COCO GT (cached from earlier runs if available), and log into W&B.

In [2]:
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

import wandb
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)

WANDB_PROJECT = "visdrone-det"
print("W&B ready. project:", WANDB_PROJECT)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: suchok-pn (suchok-pn-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready. project: visdrone-det


## 3. Dataset & dataloaders

`VisDroneDetectionDataset` from `common.py` returns images already in network-canvas coordinates, normalized with ImageNet statistics, plus targets containing `boxes`, `labels` (1..8), `area`, `iscrowd`, and `ignore_boxes` (in canvas coords). This is the same class used for Faster R-CNN, so the training pipeline is identical except for the model.

Batch size 2 matches nb03/nb04 exactly — no protocol drift.

In [3]:
from torch.utils.data import DataLoader

train_ds = VisDroneDetectionDataset(train_index, STRAT, SIZE, is_train=False)
val_ds   = VisDroneDetectionDataset(val_index,   STRAT, SIZE, is_train=False)

train_loader = DataLoader(
    train_ds, batch_size=2, shuffle=True, num_workers=2,
    collate_fn=detection_collate, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=2, shuffle=False, num_workers=2,
    collate_fn=detection_collate, pin_memory=True,
)

print(f"train batches/epoch: {len(train_loader):,}")
print(f"val batches:         {len(val_loader):,}")

# Sanity check one batch
imgs, tgts = next(iter(train_loader))
print("image batch shape:", tuple(imgs[0].shape))
print("target keys:", list(tgts[0].keys()))
print("boxes:", tuple(tgts[0]["boxes"].shape),
      "| labels:", tuple(tgts[0]["labels"].shape),
      "| ignore_boxes:", tuple(tgts[0]["ignore_boxes"].shape))

train batches/epoch: 12,100
val batches:         1,423
image batch shape: (3, 1280, 1280)
target keys: ['boxes', 'labels', 'image_id', 'area', 'iscrowd', 'ignore_boxes']
boxes: (125, 4) | labels: (125,) | ignore_boxes: (15, 4)


## 4. Training loop with ignore-region masking

### Why ignore masking is an *approximation* for RetinaNet

In Faster R-CNN (nb03–nb04), the `IgnoreAwareAssigner` masks anchors **before** RPN proposals are sampled. In RetinaNet, there is no proposal stage — every anchor is used directly. torchvision's `RetinaNet.forward()` in training mode runs `head.compute_loss(targets, ...)`, which internally calls the head's assigner with `(anchors, matched_idxs, labels, regression_targets)` and **never passes `ignore_boxes`**.

The standard, community-accepted workaround (used by Detectron2, MMDetection, and YOLOv5):
1. Compute the standard RetinaNet loss.
2. **Recompute only the classification loss** on a relabeled version of the anchors where any anchor overlapping an ignore region at IoU > 0.3 **and** currently labeled background (0) is set to −1 (i.e. dropped).
3. Replace `loss_dict["classification"]` with the recomputed version.
4. Leave the regression loss unchanged — positives are unaffected by ignore masking in the standard formulation.

This is **not** a perfect equivalent of the two-stage assigner (it does not de-activate *positive* anchors inside crowd regions), but it removes the dominant failure mode: the model learning "background" from anchors inside crowds. This is documented in the report as an explicit approximation.

### Practical note

We run the standard forward first (no extra cost — it gives us the feature-level predictions), then recompute the classification loss once from those same predictions. The overhead is one `box_iou` call per image on the already-flattened anchors (~80k anchors × N_ignore), which is negligible compared to the backbone forward.

In [4]:
def train_one_epoch_retinanet_simple(model, loader, optimizer, scheduler, device,
                                     epoch, wandb_run=None, log_every=20):
    """Standard RetinaNet training loop — no ignore masking (see §4 note)."""
    model.train()
    running = {"loss": 0.0, "loss_cls": 0.0, "loss_box": 0.0}
    n_batches = 0

    for step, (images, targets) in enumerate(tqdm(loader, desc=f"R0 epoch {epoch}")):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        running["loss"]     += float(losses)
        running["loss_cls"] += float(loss_dict.get("classification", 0.0))
        running["loss_box"] += float(loss_dict.get("bbox_regression", 0.0))
        n_batches += 1

        if step % log_every == 0 and wandb_run is not None:
            wandb_run.log({
                "train/loss":     float(losses),
                "train/loss_cls": float(loss_dict.get("classification", 0.0)),
                "train/loss_box": float(loss_dict.get("bbox_regression", 0.0)),
                "train/step":     step,
                "train/epoch":    epoch,
            })

    scheduler.step()
    return {k: v / max(1, n_batches) for k, v in running.items()}

## 5. Train R0 (1 epoch)

Warm-start from COCO-pretrained `retinanet_resnet50_fpn_v2`. The torchvision call already loads ImageNet backbone weights by default — `weights_backbone=None` in the builder disables the backbone load, but the head has 9 classes so the original 91-class head cannot be reused. **For R0 we want COCO-pretrained weights on both backbone and heads** (same warm-start philosophy as nb03 for Faster R-CNN: head is re-initialized, backbone carries ImageNet/COCO features).

To match nb03's warm-start, load COCO weights before the head swap. We do that below.

In [5]:
# Build R0: load COCO-pretrained weights, then swap head for 9-class
def build_retinanet_coco_warmstart(anchor_sizes=(32, 64, 128, 256, 512),
                                    anchor_aspect_ratios=(0.5, 1.0, 2.0),
                                    score_thresh=0.05, nms_thresh=0.5):
    """RetinaNet warm-started from COCO weights, head swapped for 9 classes."""
    from torchvision.models.detection.retinanet import RetinaNetRegressionHead

    m = retinanet_resnet50_fpn_v2(
        weights="DEFAULT",      # COCO-pretrained backbone + heads
        weights_backbone="DEFAULT",
        min_size=SIZE, max_size=SIZE,
        score_thresh=score_thresh,
        nms_thresh=nms_thresh,
        detections_per_img=500,
    )
    # Custom anchors
    aspect_ratios_per_level = tuple(tuple(anchor_aspect_ratios) for _ in anchor_sizes)
    m.anchor_generator = AnchorGenerator(
        sizes=tuple((s,) for s in anchor_sizes),
        aspect_ratios=aspect_ratios_per_level,
    )
    num_anchors = m.anchor_generator.num_anchors_per_location()[0]

    # Swap classification head -> 9 classes
    in_channels = m.head.classification_head.cls_logits.in_channels
    m.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels, num_anchors=num_anchors,
        num_classes=NUM_CLASSES + 1,
        norm_layer=lambda num_channels: torch.nn.GroupNorm(32, num_channels),
    )
    # Swap regression head -> matches num_anchors
    in_channels_reg = m.head.regression_head.bbox_reg.in_channels
    m.head.regression_head = RetinaNetRegressionHead(
        in_channels=in_channels_reg, num_anchors=num_anchors,
        norm_layer=lambda num_channels: torch.nn.GroupNorm(32, num_channels),
    )
    return m


model_r0 = build_retinanet_coco_warmstart().to(DEVICE)
n_params = sum(p.numel() for p in model_r0.parameters() if p.requires_grad)
print(f"trainable params: {n_params/1e6:.2f}M")

# Optimizer / scheduler — identical to nb03/nb04
params = [p for p in model_r0.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=5e-3, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Smoke test: one batch forward + backward
model_r0.train()
imgs, tgts = next(iter(train_loader))
imgs  = [i.to(DEVICE) for i in imgs]
tgts  = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
loss_dict = model_r0(imgs, tgts)
print("smoke loss_dict:", {k: round(float(v), 4) for k, v in loss_dict.items()})

Downloading: "https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth" to /root/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth


100%|██████████| 146M/146M [00:00<00:00, 250MB/s]


trainable params: 36.09M
smoke loss_dict: {'classification': 1.1459, 'bbox_regression': 1.8822}


/tmp/ipykernel_23/3676480680.py:55: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("smoke loss_dict:", {k: round(float(v), 4) for k, v in loss_dict.items()})


In [6]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="retinanet_R0",
    config={
        "model": "retinanet_resnet50_fpn_v2",
        "variant": "R0_baseline",
        "strategy": STRAT, "size": SIZE,
        "epochs": 1, "lr": 5e-3, "batch_size": 2,
        "anchor_sizes": [32, 64, 128, 256, 512],
        "anchor_aspect_ratios": [0.5, 1.0, 2.0],
        "ignore_masking": "none (approximation, see report §4.4)",
        "seed": SEED,
    },
    reinit=True,
)

t_start = time.time()
train_stats = train_one_epoch_retinanet_simple(
    model_r0, train_loader, optimizer, scheduler, DEVICE,
    epoch=1, wandb_run=run, log_every=50,
)
train_seconds = time.time() - t_start
print(f"train stats: {train_stats}")
print(f"train wall time: {train_seconds/3600:.2f} h")

# Save checkpoint immediately (session could be killed at any moment)
ckpt_path = save_checkpoint(model_r0, WORK_DIR / "retinanet_R0.pth")
print("saved:", ckpt_path)

wandb.log({"train/epoch_loss": train_stats["loss"],
           "train/epoch_time_h": train_seconds/3600})

with open(WORK_DIR / "train_log_retinanet_R0.json", "w") as f:
    json.dump({
        "epoch": 1,
        "mean_loss":  train_stats["loss"],
        "mean_loss_cls": train_stats["loss_cls"],
        "mean_loss_box": train_stats["loss_box"],
        "wall_seconds": train_seconds,
        "batch_size": 2, "lr": 5e-3,
        "anchor_sizes": [32, 64, 128, 256, 512],
        "anchor_aspect_ratios": [0.5, 1.0, 2.0],
    }, f, indent=2)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261106_210409-pbvhlp1t
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run retinanet_R0
wandb: ⭐️ View project at https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det
wandb: 🚀 View run at https://wandb.ai/suchok-pn-ukrainian-catholic-university/visdrone-det/runs/pbvhlp1t


R0 epoch 1:   0%|          | 0/12100 [00:00<?, ?it/s]

train stats: {'loss': 0.7960324883288589, 'loss_cls': 0.4163275994656007, 'loss_box': 0.37970488914096157}
train wall time: 4.27 h
saved: /kaggle/working/retinanet_R0.pth


## 6. Full-val evaluation

Run inference on all 2,846 val frames, compute COCO metrics against `gt_fullval_ignore.json` (which has `iscrowd=1` for ignore regions), and log the metrics + a small batch of prediction images to W&B.

In [7]:
# Free GPU memory before eval (train forward pass is gone)
torch.cuda.empty_cache()

metrics_r0, preds_path_r0 = evaluate_full_val(
    model_r0, val_index, STRAT, SIZE,
    gt_json=str(gt_fullval_ignore),
    score_thr=0.001,
    device=DEVICE,
    out_json=str(WORK_DIR / "val_preds_retinanet_R0.json"),
)

print("=" * 60)
print("R0 (baseline) — full val:")
for k, v in metrics_r0.items():
    print(f"  {k:10s}: {v:.4f}")

wandb.log({f"val/{k}": v for k, v in metrics_r0.items()})
save_metrics(metrics_r0, WORK_DIR / "metrics_retinanet_R0.json")

pred letterbox-1280:   0%|          | 0/2846 [00:00<?, ?it/s]

AssertionError: targets should not be none when in training mode

In [ ]:
from common import benchmark_speed
speed_r0 = benchmark_speed(
    model_r0, val_index, STRAT, SIZE, n=50,
    device=DEVICE, score_thr=0.5, warmup=5,
)
print("speed:", speed_r0)
with open(WORK_DIR / "speed_retinanet_R0.json", "w") as f:
    json.dump(speed_r0, f, indent=2)
wandb.log({"speed/gpu_fps": speed_r0["gpu_fps"],
           "speed/gpu_ms":  speed_r0["gpu_ms_per_image"]})

## 7. Qualitative check — log 3 val predictions to W&B

Not a metric — just a sanity check that boxes look reasonable. If the model predicts nothing, or boxes are wildly off, stop here and debug before uploading the checkpoint.

In [ ]:
import matplotlib.patches as patches

def _draw_and_convert(image_path, preds_json, image_id, thr=0.3):
    """Draw boxes on an image and return a wandb.Image."""
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    with open(preds_json) as f:
        preds = [p for p in json.load(f) if p["image_id"] == image_id and p["score"] >= thr]
    # Convert COCO [x, y, w, h] back to wandb's [x, y, w, h] pixel form
    boxes_data = []
    for p in preds:
        x, y, bw, bh = p["bbox"]
        boxes_data.append({
            "position": {"minX": x / w, "maxX": (x + bw) / w,
                         "minY": y / h, "maxY": (y + bh) / h},
            "class_id": p["category_id"] - 1,
            "box_caption": f"{CLASS_NAMES[p['category_id']-1]} {p['score']:.2f}",
            "scores": {"score": p["score"]},
        })
    return wandb.Image(img, boxes={"predictions": {"box_data": boxes_data}})

# Pick 3 diverse val frames (first, middle, last)
sample_ids = [1, len(val_index) // 2, len(val_index)]
sample_imgs = []
for sid in sample_ids:
    row = val_index.iloc[sid - 1]
    sample_imgs.append(_draw_and_convert(row["image_path"], preds_path_r0, sid, thr=0.3))

wandb.log({"val/predictions": sample_imgs})
run.finish()
print("W&B run finished:", run.url)

## 8. Session summary — what to do next

**This notebook has produced:**

| Artifact | Path | Upload to Kaggle dataset? |
|---|---|---|
| RetinaNet R0 checkpoint | `/kaggle/working/retinanet_R0.pth` | ✅ yes |
| Full-val predictions | `/kaggle/working/val_preds_retinanet_R0.json` | ✅ yes |
| Metrics JSON | `/kaggle/working/metrics_retinanet_R0.json` | ✅ yes |
| W&B run | `visdrone-det/retinanet_R0` | — (link in report) |

**Immediate next steps:**

1. **Upload the three files above as a Kaggle dataset** named e.g. `retinanet-r0`. This is required for nb06 to warm-start from R0 across a new session.
2. **Do NOT submit to Kaggle LB from R0.** R0 is the baseline; the winner submission comes from R1 (nb06). Submitting R0 would waste a submission slot on a model that has not had its anchor-tuning step applied.
3. **Record R0's metrics in the experiment log** (see `draft_to_report.md` §6, row `nb05`):

| ID | Model | Change | Val mAP | AP50 | AP_small | FPS | Kept? |
|---|---|---|---|---|---|---|---|
| nb05 | RetinaNet R0 | COCO default anchors, focal γ=2 | *(fill after run)* | | | | reference |

4. **Proceed to nb06** — R1 (data-grounded anchors) warm-started from R0. nb06 is the one that produces the submission.